In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "413f0df2",
   "metadata": {},
   "source": [
    "# 03 — Preprocessing experiments: stopwords & lemmatisation\n",
    "\n",
    "**CI1 — Machine Learning mini-project · dataset: TweetEval – Offensive**\n",
    "(`0` = non-offensive, `1` = offensive)\n",
    "\n",
    "| | |\n",
    "|---|---|\n",
    "| **Group** | Youssef · Emna · *&lt;member 3&gt;* · *&lt;member 4&gt;*  ← *(fill in the two missing names)* |\n",
    "| **Notebook owner** | **Youssef** — preprocessing experiments & class balancing |\n",
    "| **Input** | `data/processed/train_clean.csv`, `data/processed/val_clean.csv` (cleaning = Emna, notebook 02) |\n",
    "| **Output** | `results/metrics/03_preprocessing_experiments/preprocessing_comparison.csv` + 4 figures |\n",
    "| **Test set** | **never loaded in this notebook** — it is used once, for the final model |\n",
    "\n",
    "### The question this notebook answers with numbers\n",
    "> Do stopword removal and lemmatisation actually improve the classifier on this dataset?\n",
    "\n",
    "Emna's cleaning is deliberately *basic* (lowercase, URLs, `@mention → user`, `#` removed,\n",
    "punctuation/emoji removed, apostrophes and masked profanity kept). We compare **5 variants of the\n",
    "text** under **identical conditions** and keep only what improves the validation **F1 of the\n",
    "offensive class** — not accuracy, because with ~2 non-offensive tweets for 1 offensive tweet,\n",
    "always answering *non-offensive* already gives ≈ 0.65 accuracy."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "b82c331b",
   "metadata": {},
   "source": [
    "## 0 · Setup\n",
    "\n",
    "Paths work whether you launch the notebook from the repository root or from `notebooks/`.\n",
    "The NLTK corpora are downloaded once (no-op if already present)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "3887c8db",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:06.695569Z",
     "iopub.status.busy": "2026-11-10T18:55:06.695197Z",
     "iopub.status.idle": "2026-11-10T18:55:08.258994Z",
     "shell.execute_reply": "2026-11-10T18:55:08.258363Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "PROJECT_ROOT : /home/user/repo\n",
      "data/processed exists : True\n"
     ]
    }
   ],
   "source": [
    "import sys, time, html, re, warnings\n",
    "from pathlib import Path\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "warnings.filterwarnings(\"ignore\", category=FutureWarning)\n",
    "\n",
    "# --- paths: work from the repo root or from notebooks/ -----------------------\n",
    "PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == \"notebooks\" else Path.cwd()\n",
    "if str(PROJECT_ROOT) not in sys.path:\n",
    "    sys.path.insert(0, str(PROJECT_ROOT))\n",
    "\n",
    "PROC_DIR = PROJECT_ROOT / \"data\" / \"processed\"\n",
    "RAW_DIR  = PROJECT_ROOT / \"data\" / \"raw\"\n",
    "\n",
    "# --- NLTK data: download once, silently --------------------------------------\n",
    "import nltk\n",
    "for package in (\"stopwords\", \"wordnet\", \"omw-1.4\"):\n",
    "    try:\n",
    "        nltk.data.find(f\"corpora/{package}\")\n",
    "    except LookupError:\n",
    "        nltk.download(package, quiet=True)\n",
    "\n",
    "print(\"PROJECT_ROOT :\", PROJECT_ROOT)\n",
    "print(\"data/processed exists :\", PROC_DIR.is_dir())"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "74edaa40",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:08.260830Z",
     "iopub.status.busy": "2026-11-10T18:55:08.260581Z",
     "iopub.status.idle": "2026-11-10T18:55:08.265312Z",
     "shell.execute_reply": "2026-11-10T18:55:08.264717Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "results/metrics/03_preprocessing_experiments\n",
      "results/figures/03_preprocessing_experiments\n"
     ]
    }
   ],
   "source": [
    "# --- output folders: one sub-folder per notebook (guide, section 6) ---------\n",
    "METRIC_DIR = PROJECT_ROOT / \"results\" / \"metrics\" / \"03_preprocessing_experiments\"\n",
    "FIGURE_DIR = PROJECT_ROOT / \"results\" / \"figures\" / \"03_preprocessing_experiments\"\n",
    "for folder in (METRIC_DIR, FIGURE_DIR):\n",
    "    folder.mkdir(parents=True, exist_ok=True)\n",
    "print(METRIC_DIR.relative_to(PROJECT_ROOT))\n",
    "print(FIGURE_DIR.relative_to(PROJECT_ROOT))\n",
    "\n",
    "sns.set_theme(style=\"whitegrid\", palette=\"deep\")\n",
    "plt.rcParams[\"figure.dpi\"] = 110"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "46b5fdc0",
   "metadata": {},
   "source": [
    "## 1 · The data we start from (Emna's output, notebook 02)\n",
    "\n",
    "`text` = original tweet, `text_clean` = Emna's cleaned tweet, `label` = ground truth.\n",
    "We **never** touch the labels and we **never** load the test split here.\n",
    "\n",
    "`keep_default_na=False` matters: a cleaned tweet can be an empty string, and without this option\n",
    "pandas would turn it into `NaN`, which makes `TfidfVectorizer` crash."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "b7a4e789",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:08.267059Z",
     "iopub.status.busy": "2026-11-10T18:55:08.266910Z",
     "iopub.status.idle": "2026-11-10T18:55:08.273670Z",
     "shell.execute_reply": "2026-11-10T18:55:08.273073Z"
    }
   },
   "outputs": [],
   "source": [
    "# ---------------------------------------------------------------------------\n",
    "# Temporary bridge: Emna's data/processed/*_clean.csv are not in the repository\n",
    "# yet (data/processed/ is git-ignored). If they are missing, this cell rebuilds\n",
    "# them from data/raw using the cleaning described in her notebook 02, so that\n",
    "# this notebook is runnable by anyone. Delete the function once her files are\n",
    "# committed - the rest of the notebook does not change.\n",
    "# ---------------------------------------------------------------------------\n",
    "def clean_text_like_emna(text: str) -> str:\n",
    "    \"\"\"lowercase · URLs out · @mention -> 'user' · '#' out · punctuation/emoji out\n",
    "    · apostrophes of contractions kept (don't) · masked profanity kept (f***)\"\"\"\n",
    "    t = html.unescape(str(text))\n",
    "    t = t.replace(\"\\u2019\", \"'\").replace(\"\\u2018\", \"'\").replace(\"`\", \"'\")\n",
    "    t = re.sub(r\"https?://\\S+|www\\.\\S+\", \" \", t, flags=re.I).lower()\n",
    "    t = re.sub(r\"@\\w+\", \" user \", t)\n",
    "    t = re.sub(r\"#(\\w+)\", r\"\\1\", t)\n",
    "    t = re.sub(r\"[^a-z0-9'*\\s]\", \" \", t)\n",
    "    t = re.sub(r\"(?<![a-z])'+|'+(?![a-z])\", \" \", t)      # orphan apostrophes\n",
    "    return re.sub(r\"\\s+\", \" \", t).strip()\n",
    "\n",
    "\n",
    "def load_clean_splits():\n",
    "    \"\"\"Load train_clean.csv / val_clean.csv (rebuild from data/raw if absent).\n",
    "\n",
    "    The test split is deliberately NOT loaded nor rebuilt: it must stay\n",
    "    untouched until the final evaluation.\n",
    "    \"\"\"\n",
    "    files = {\"train\": PROC_DIR / \"train_clean.csv\", \"val\": PROC_DIR / \"val_clean.csv\"}\n",
    "\n",
    "    if not all(f.is_file() for f in files.values()):\n",
    "        from src.data_loader import load_dataset\n",
    "        print(\"data/processed/ is incomplete -> rebuilding train/val from data/raw (temporary)\")\n",
    "        PROC_DIR.mkdir(parents=True, exist_ok=True)\n",
    "        splits = load_dataset(str(RAW_DIR))\n",
    "        for split, key in ((\"train\", \"train\"), (\"val\", \"validation\")):\n",
    "            df = splits[key].copy()\n",
    "            df[\"text_clean\"] = df[\"text\"].map(clean_text_like_emna)\n",
    "            df = df[[\"text\", \"text_clean\", \"label\"]]\n",
    "            df.to_csv(files[split], index=False, encoding=\"utf-8\")\n",
    "            print(f\"  wrote {files[split].relative_to(PROJECT_ROOT)}  {df.shape}\")\n",
    "\n",
    "    frames = {name: pd.read_csv(path, keep_default_na=False)   # keep_default_na=False: empty strings stay \"\"\n",
    "              for name, path in files.items()}\n",
    "    for name, df in frames.items():\n",
    "        assert list(df.columns[:3]) == [\"text\", \"text_clean\", \"label\"], (name, df.columns)\n",
    "        assert set(df[\"label\"].unique()) == {0, 1}, f\"unexpected labels in {name}\"\n",
    "    return frames[\"train\"], frames[\"val\"]"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "a87e75e2",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:08.275376Z",
     "iopub.status.busy": "2026-11-10T18:55:08.275198Z",
     "iopub.status.idle": "2026-11-10T18:55:08.332295Z",
     "shell.execute_reply": "2026-11-10T18:55:08.331956Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "train: (11916, 3)  val: (1324, 3)\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>text</th>\n",
       "      <th>text_clean</th>\n",
       "      <th>label</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>0</th>\n",
       "      <td>@user Bono... who cares. Soon people will unde...</td>\n",
       "      <td>user bono who cares soon people will understan...</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>1</th>\n",
       "      <td>@user Eight years the republicans denied obama...</td>\n",
       "      <td>user eight years the republicans denied obama'...</td>\n",
       "      <td>1</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>2</th>\n",
       "      <td>@user Get him some line help. He is gonna be j...</td>\n",
       "      <td>user get him some line help he is gonna be jus...</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                                                text  \\\n",
       "0  @user Bono... who cares. Soon people will unde...   \n",
       "1  @user Eight years the republicans denied obama...   \n",
       "2  @user Get him some line help. He is gonna be j...   \n",
       "\n",
       "                                          text_clean  label  \n",
       "0  user bono who cares soon people will understan...      0  \n",
       "1  user eight years the republicans denied obama'...      1  \n",
       "2  user get him some line help he is gonna be jus...      0  "
      ]
     },
     "execution_count": 4,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "train, val = load_clean_splits()\n",
    "print(\"train:\", train.shape, \" val:\", val.shape)\n",
    "train.head(3)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "id": "068ab800",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:08.333911Z",
     "iopub.status.busy": "2026-11-10T18:55:08.333775Z",
     "iopub.status.idle": "2026-11-10T18:55:08.351563Z",
     "shell.execute_reply": "2026-11-10T18:55:08.350984Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>rows</th>\n",
       "      <th>offensive_rows</th>\n",
       "      <th>pct_offensive</th>\n",
       "      <th>empty_text_clean</th>\n",
       "      <th>contains_placeholder_user_%</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>train</th>\n",
       "      <td>11916</td>\n",
       "      <td>3941</td>\n",
       "      <td>33.1</td>\n",
       "      <td>0</td>\n",
       "      <td>92.8</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>val</th>\n",
       "      <td>1324</td>\n",
       "      <td>459</td>\n",
       "      <td>34.7</td>\n",
       "      <td>0</td>\n",
       "      <td>91.8</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "        rows  offensive_rows  pct_offensive  empty_text_clean  \\\n",
       "train  11916            3941           33.1                 0   \n",
       "val     1324             459           34.7                 0   \n",
       "\n",
       "       contains_placeholder_user_%  \n",
       "train                         92.8  \n",
       "val                           91.8  "
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "majority-class baseline on validation (always 'non-offensive'): 65.3 % accuracy, F1 = 0.00 for the offensive class\n"
     ]
    }
   ],
   "source": [
    "# --- sanity checks before any experiment (identical splits for every variant) ---\n",
    "report = pd.DataFrame({\n",
    "    \"rows\": [len(train), len(val)],\n",
    "    \"offensive_rows\": [int(train.label.sum()), int(val.label.sum())],\n",
    "    \"pct_offensive\": [round(100 * train.label.mean(), 1), round(100 * val.label.mean(), 1)],\n",
    "    \"empty_text_clean\": [int((train.text_clean.str.strip() == \"\").sum()),\n",
    "                         int((val.text_clean.str.strip() == \"\").sum())],\n",
    "    \"contains_placeholder_user_%\": [round(100 * train.text_clean.str.contains(r\"\\buser\\b\").mean(), 1),\n",
    "                                    round(100 * val.text_clean.str.contains(r\"\\buser\\b\").mean(), 1)],\n",
    "}, index=[\"train\", \"val\"])\n",
    "display(report)\n",
    "print(\"majority-class baseline on validation (always 'non-offensive'):\",\n",
    "      round(100 * (1 - val.label.mean()), 1), \"% accuracy, F1 = 0.00 for the offensive class\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e121a5b2",
   "metadata": {},
   "source": [
    "## 2 · The variants we compare\n",
    "\n",
    "| Variant | Text transformation | Why it might help | Why it might hurt |\n",
    "|---|---|---|---|\n",
    "| **A** `clean_only` | nothing (Emna's `text_clean`) | reference point | vocabulary 18k tokens, lots of sparsity |\n",
    "| **B** `stopwords` | NLTK English list **minus all negations** | removes `the/and/of` noise, smaller vocabulary | `you`/`i` disappear: the *target* of the insult is lost |\n",
    "| **B2** `stopwords_keep_pron` | B **+ pronouns kept** | *\"you are an idiot\"* ≠ *\"I am an idiot\"* | slightly larger vocabulary |\n",
    "| **B3** `stopwords_drop_user` | B + the `user` placeholder dropped | the placeholder is in 93 % of tweets → no signal | some tweets become empty |\n",
    "| **C** `stopwords_lemma` | B + lemmatisation (verb→noun pass) | `fucking/fucked/fucks` share one feature | WordNet noise (`hating → hat`), ~3× slower |\n",
    "| **C2** `lemma_keep_pron` | **B2 + lemmatisation** (added by us) | combines the two wins: pronouns kept **and** inflections merged | the largest feature matrix (hits the 20k cap) |\n",
    "\n",
    "Variant **C2** is not in the guide: it was added *after* seeing that pronouns help (B2 > B) and that\n",
    "lemmatisation helps (C > B) — the natural next question is \"do both wins add up?\".\n",
    "\n",
    "**Golden rule of this notebook**: only the text transformation changes between variants.\n",
    "Same splits, same TF-IDF settings, same model, same `random_state=42`."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "id": "95d61c3b",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:08.353091Z",
     "iopub.status.busy": "2026-11-10T18:55:08.352930Z",
     "iopub.status.idle": "2026-11-10T18:55:08.361201Z",
     "shell.execute_reply": "2026-11-10T18:55:08.360599Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "NLTK English list                : 198 words\n",
      "build_stopwords()                : 177 words  (negations removed)\n",
      "build_stopwords(keep_pronouns=1) : 148 words\n",
      "build_stopwords(drop_user=True)  : 178 words  ('user' added)\n",
      "\n",
      "negations protected              : 48 forms\n",
      "  examples: ain't, aint, aren't, arent, can't, cannot, cant, couldn't, couldnt, didn't, didnt, doesn't ...\n",
      "negations that leaked into a list: none  ✔\n",
      "\n",
      "self-test : not idiot , don't even try\n",
      "B2 (pron.): you not idiot , don't even try\n"
     ]
    }
   ],
   "source": [
    "from src.stopwords_config import (NEGATIONS, build_stopwords, remove_stopwords,\n",
    "                                  stopword_stats, missing_protected_tokens, DEFAULT_STOPWORDS)\n",
    "from src.lemmatization import lemmatize_text, lemmatization_stats\n",
    "\n",
    "SW_DEFAULT = build_stopwords()                         # B\n",
    "SW_PRON    = build_stopwords(keep_pronouns=True)       # B2\n",
    "SW_NOUSER  = build_stopwords(drop_user=True)           # B3\n",
    "\n",
    "print(f\"NLTK English list                : 198 words\")\n",
    "print(f\"build_stopwords()                : {len(SW_DEFAULT)} words  (negations removed)\")\n",
    "print(f\"build_stopwords(keep_pronouns=1) : {len(SW_PRON)} words\")\n",
    "print(f\"build_stopwords(drop_user=True)  : {len(SW_NOUSER)} words  ('user' added)\")\n",
    "print()\n",
    "print(f\"negations protected              : {len(NEGATIONS)} forms\")\n",
    "print(\"  examples:\", \", \".join(sorted(NEGATIONS)[:12]), \"...\")\n",
    "print(\"negations that leaked into a list:\", missing_protected_tokens(SW_DEFAULT) or \"none  ✔\")\n",
    "print()\n",
    "print(\"self-test :\", remove_stopwords(\"you are not an idiot , don't even try\"))\n",
    "print(\"B2 (pron.):\", remove_stopwords(\"you are not an idiot , don't even try\", SW_PRON))"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "9c82dc18",
   "metadata": {},
   "source": [
    "## 3 · Before / after: proving the meaning is not destroyed\n",
    "\n",
    "Automated metrics cannot tell you that a transformation destroyed a sentence — you have to look.\n",
    "The examples below are picked **by content** (a negation, a contraction, masked profanity, a\n",
    "tweet that is only mentions, an inflected insult, a past tense) so that they are reproducible."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "id": "c659b295",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:08.362941Z",
     "iopub.status.busy": "2026-11-10T18:55:08.362811Z",
     "iopub.status.idle": "2026-11-10T18:55:10.610208Z",
     "shell.execute_reply": "2026-11-10T18:55:10.609715Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead tr th {\n",
       "        text-align: left;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr>\n",
       "      <th>why</th>\n",
       "      <th>negation 'not'</th>\n",
       "      <th>contraction don't</th>\n",
       "      <th>masked profanity</th>\n",
       "      <th>only mentions</th>\n",
       "      <th>inflected insult</th>\n",
       "      <th>past tense 'was'</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>id</th>\n",
       "      <th>19</th>\n",
       "      <th>72</th>\n",
       "      <th>2949</th>\n",
       "      <th>18</th>\n",
       "      <th>204</th>\n",
       "      <th>30</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>A_clean_only</th>\n",
       "      <td>great comedy there cnn news not so much maga licensereporters</td>\n",
       "      <td>user tell that to him he is very perfectionist i don't think we ...</td>\n",
       "      <td>user biggest corrupt lying piece of s*** there ever was</td>\n",
       "      <td>user user you mean the way hillary re victimized all of bills ac...</td>\n",
       "      <td>user user user user hes comin for ur woolly bitches</td>\n",
       "      <td>user i was think she jisoo but i'm focused i found she is taeyoun</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B_stopwords</th>\n",
       "      <td>great comedy cnn news not much maga licensereporters</td>\n",
       "      <td>user tell perfectionist don't think change ever</td>\n",
       "      <td>user biggest corrupt lying piece s*** ever</td>\n",
       "      <td>user user mean way hillary victimized bills accusers</td>\n",
       "      <td>user user user user hes comin ur woolly bitches</td>\n",
       "      <td>user think jisoo focused found taeyoun</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B2_keep_pron</th>\n",
       "      <td>great comedy cnn news not much maga licensereporters</td>\n",
       "      <td>user tell him he perfectionist i don't think we change ever</td>\n",
       "      <td>user biggest corrupt lying piece s*** ever</td>\n",
       "      <td>user user you mean way hillary victimized bills accusers</td>\n",
       "      <td>user user user user hes comin ur woolly bitches</td>\n",
       "      <td>user i think she jisoo focused i found she taeyoun</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>C_stop+lemma</th>\n",
       "      <td>great comedy cnn news not much maga licensereporters</td>\n",
       "      <td>user tell perfectionist don't think change ever</td>\n",
       "      <td>user biggest corrupt lie piece s*** ever</td>\n",
       "      <td>user user mean way hillary victimize bill accuser</td>\n",
       "      <td>user user user user he comin ur woolly bitch</td>\n",
       "      <td>user think jisoo focus find taeyoun</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>C2_lemma+pron</th>\n",
       "      <td>great comedy cnn news not much maga licensereporters</td>\n",
       "      <td>user tell him he perfectionist i don't think we change ever</td>\n",
       "      <td>user biggest corrupt lie piece s*** ever</td>\n",
       "      <td>user user you mean way hillary victimize bill accuser</td>\n",
       "      <td>user user user user he comin ur woolly bitch</td>\n",
       "      <td>user i think she jisoo focus i find she taeyoun</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "why                                                           negation 'not'  \\\n",
       "id                                                                      19     \n",
       "A_clean_only   great comedy there cnn news not so much maga licensereporters   \n",
       "B_stopwords             great comedy cnn news not much maga licensereporters   \n",
       "B2_keep_pron            great comedy cnn news not much maga licensereporters   \n",
       "C_stop+lemma            great comedy cnn news not much maga licensereporters   \n",
       "C2_lemma+pron           great comedy cnn news not much maga licensereporters   \n",
       "\n",
       "why                                                              contraction don't  \\\n",
       "id                                                                            72     \n",
       "A_clean_only   user tell that to him he is very perfectionist i don't think we ...   \n",
       "B_stopwords                        user tell perfectionist don't think change ever   \n",
       "B2_keep_pron           user tell him he perfectionist i don't think we change ever   \n",
       "C_stop+lemma                       user tell perfectionist don't think change ever   \n",
       "C2_lemma+pron          user tell him he perfectionist i don't think we change ever   \n",
       "\n",
       "why                                                   masked profanity  \\\n",
       "id                                                                2949   \n",
       "A_clean_only   user biggest corrupt lying piece of s*** there ever was   \n",
       "B_stopwords                 user biggest corrupt lying piece s*** ever   \n",
       "B2_keep_pron                user biggest corrupt lying piece s*** ever   \n",
       "C_stop+lemma                  user biggest corrupt lie piece s*** ever   \n",
       "C2_lemma+pron                 user biggest corrupt lie piece s*** ever   \n",
       "\n",
       "why                                                                  only mentions  \\\n",
       "id                                                                            18     \n",
       "A_clean_only   user user you mean the way hillary re victimized all of bills ac...   \n",
       "B_stopwords                   user user mean way hillary victimized bills accusers   \n",
       "B2_keep_pron              user user you mean way hillary victimized bills accusers   \n",
       "C_stop+lemma                     user user mean way hillary victimize bill accuser   \n",
       "C2_lemma+pron                user user you mean way hillary victimize bill accuser   \n",
       "\n",
       "why                                               inflected insult  \\\n",
       "id                                                            204    \n",
       "A_clean_only   user user user user hes comin for ur woolly bitches   \n",
       "B_stopwords        user user user user hes comin ur woolly bitches   \n",
       "B2_keep_pron       user user user user hes comin ur woolly bitches   \n",
       "C_stop+lemma          user user user user he comin ur woolly bitch   \n",
       "C2_lemma+pron         user user user user he comin ur woolly bitch   \n",
       "\n",
       "why                                                             past tense 'was'  \n",
       "id                                                                          30    \n",
       "A_clean_only   user i was think she jisoo but i'm focused i found she is taeyoun  \n",
       "B_stopwords                               user think jisoo focused found taeyoun  \n",
       "B2_keep_pron                  user i think she jisoo focused i found she taeyoun  \n",
       "C_stop+lemma                                 user think jisoo focus find taeyoun  \n",
       "C2_lemma+pron                    user i think she jisoo focus i find she taeyoun  "
      ]
     },
     "execution_count": 7,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "# deterministic, content-based selection of illustrative tweets\n",
    "PROBES = {\n",
    "    \"negation 'not'\":      lambda t: \" not \" in f\" {t} \",\n",
    "    \"contraction don't\":   lambda t: \"don't\" in t,\n",
    "    \"masked profanity\":    lambda t: \"***\" in t,\n",
    "    \"only mentions\":       lambda t: t.replace(\"user\", \"\").strip() != \"\" and t.count(\"user\") >= 3,\n",
    "    \"inflected insult\":    lambda t: \"idiots\" in t or \"bitches\" in t,\n",
    "    \"past tense 'was'\":    lambda t: \" was \" in f\" {t} \",\n",
    "}\n",
    "rows, used = [], set()\n",
    "for label, probe in PROBES.items():\n",
    "    for idx, text in train.text_clean.items():\n",
    "        if idx not in used and probe(text) and 6 <= len(text.split()) <= 22:\n",
    "            used.add(idx)\n",
    "            rows.append({\"why\": label, \"id\": idx, \"A_clean_only\": text,\n",
    "                         \"B_stopwords\": remove_stopwords(text),\n",
    "                         \"B2_keep_pron\": remove_stopwords(text, SW_PRON),\n",
    "                         \"C_stop+lemma\": lemmatize_text(remove_stopwords(text)),\n",
    "                         \"C2_lemma+pron\": lemmatize_text(remove_stopwords(text, SW_PRON))})\n",
    "            break\n",
    "\n",
    "examples = pd.DataFrame(rows).set_index([\"why\", \"id\"])\n",
    "examples.to_csv(METRIC_DIR / \"before_after_examples.csv\")      # traceability of the figures below\n",
    "pd.set_option(\"display.max_colwidth\", 68)\n",
    "examples.T"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "4bc88692",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:10.611780Z",
     "iopub.status.busy": "2026-11-10T18:55:10.611643Z",
     "iopub.status.idle": "2026-11-10T18:55:10.851470Z",
     "shell.execute_reply": "2026-11-10T18:55:10.850944Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>val tweets containing 'not'</th>\n",
       "      <th>tokens 'not'</th>\n",
       "      <th>tokens \"don't\"</th>\n",
       "      <th>tokens 'user'</th>\n",
       "      <th>'not' preserved</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>variant</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>A_clean_only</th>\n",
       "      <td>155</td>\n",
       "      <td>171</td>\n",
       "      <td>99</td>\n",
       "      <td>3455</td>\n",
       "      <td>✔</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B_stopwords</th>\n",
       "      <td>155</td>\n",
       "      <td>171</td>\n",
       "      <td>99</td>\n",
       "      <td>3455</td>\n",
       "      <td>✔</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B2_keep_pron</th>\n",
       "      <td>155</td>\n",
       "      <td>171</td>\n",
       "      <td>99</td>\n",
       "      <td>3455</td>\n",
       "      <td>✔</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B3_drop_user</th>\n",
       "      <td>155</td>\n",
       "      <td>171</td>\n",
       "      <td>99</td>\n",
       "      <td>0</td>\n",
       "      <td>✔</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>C_stop+lemma</th>\n",
       "      <td>155</td>\n",
       "      <td>171</td>\n",
       "      <td>99</td>\n",
       "      <td>3456</td>\n",
       "      <td>✔</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>C2_lemma+pron</th>\n",
       "      <td>155</td>\n",
       "      <td>171</td>\n",
       "      <td>99</td>\n",
       "      <td>3456</td>\n",
       "      <td>✔</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "               val tweets containing 'not'  tokens 'not'  tokens \"don't\"  \\\n",
       "variant                                                                    \n",
       "A_clean_only                           155           171              99   \n",
       "B_stopwords                            155           171              99   \n",
       "B2_keep_pron                           155           171              99   \n",
       "B3_drop_user                           155           171              99   \n",
       "C_stop+lemma                           155           171              99   \n",
       "C2_lemma+pron                          155           171              99   \n",
       "\n",
       "               tokens 'user' 'not' preserved  \n",
       "variant                                       \n",
       "A_clean_only            3455               ✔  \n",
       "B_stopwords             3455               ✔  \n",
       "B2_keep_pron            3455               ✔  \n",
       "B3_drop_user               0               ✔  \n",
       "C_stop+lemma            3456               ✔  \n",
       "C2_lemma+pron           3456               ✔  "
      ]
     },
     "execution_count": 8,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "# --- quantitative proof that the negations survive every variant -----------\n",
    "def count_token(texts, token):\n",
    "    return sum(t.split().count(token) for t in texts)\n",
    "\n",
    "check = []\n",
    "for name, fn, sw in [(\"A_clean_only\", lambda t: t, None),\n",
    "                     (\"B_stopwords\", remove_stopwords, SW_DEFAULT),\n",
    "                     (\"B2_keep_pron\", lambda t: remove_stopwords(t, SW_PRON), SW_PRON),\n",
    "                     (\"B3_drop_user\", lambda t: remove_stopwords(t, SW_NOUSER), SW_NOUSER),\n",
    "                     (\"C_stop+lemma\", lambda t: lemmatize_text(remove_stopwords(t)), SW_DEFAULT),\n",
    "                     (\"C2_lemma+pron\", lambda t: lemmatize_text(remove_stopwords(t, SW_PRON)), SW_PRON)]:\n",
    "    transformed = val.text_clean.map(fn)\n",
    "    check.append({\n",
    "        \"variant\": name,\n",
    "        \"val tweets containing 'not'\": int(transformed.str.contains(r\"\\bnot\\b\").sum()),\n",
    "        \"tokens 'not'\": count_token(transformed, \"not\"),\n",
    "        \"tokens \\\"don't\\\"\": count_token(transformed, \"don't\"),\n",
    "        \"tokens 'user'\": count_token(transformed, \"user\"),\n",
    "        \"'not' preserved\": \"✔\" if count_token(transformed, \"not\") == count_token(val.text_clean, \"not\") else \"✘ LOST\",\n",
    "    })\n",
    "negation_check = pd.DataFrame(check).set_index(\"variant\")\n",
    "negation_check"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 9,
   "id": "2a3f6a3e",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:10.852954Z",
     "iopub.status.busy": "2026-11-10T18:55:10.852827Z",
     "iopub.status.idle": "2026-11-10T18:55:12.835045Z",
     "shell.execute_reply": "2026-11-10T18:55:12.834635Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>avg_words_train</th>\n",
       "      <th>vocab_train</th>\n",
       "      <th>empty_train</th>\n",
       "      <th>empty_val</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>variant</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>A_clean_only</th>\n",
       "      <td>21.94</td>\n",
       "      <td>18463</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B_stopwords</th>\n",
       "      <td>13.15</td>\n",
       "      <td>18300</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B2_keep_pron</th>\n",
       "      <td>15.05</td>\n",
       "      <td>18329</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B3_drop_user</th>\n",
       "      <td>10.63</td>\n",
       "      <td>18299</td>\n",
       "      <td>58</td>\n",
       "      <td>6</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>C_stop+lemma</th>\n",
       "      <td>13.15</td>\n",
       "      <td>14679</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>C2_lemma+pron</th>\n",
       "      <td>15.05</td>\n",
       "      <td>14706</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "               avg_words_train  vocab_train  empty_train  empty_val\n",
       "variant                                                            \n",
       "A_clean_only             21.94        18463            0          0\n",
       "B_stopwords              13.15        18300            0          0\n",
       "B2_keep_pron             15.05        18329            0          0\n",
       "B3_drop_user             10.63        18299           58          6\n",
       "C_stop+lemma             13.15        14679            0          0\n",
       "C2_lemma+pron            15.05        14706            0          0"
      ]
     },
     "execution_count": 9,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "# --- what each transformation costs: tokens, empties, vocabulary ----------\n",
    "stats_rows = []\n",
    "for name, fn in [(\"A_clean_only\", lambda t: t),\n",
    "                 (\"B_stopwords\", remove_stopwords),\n",
    "                 (\"B2_keep_pron\", lambda t: remove_stopwords(t, SW_PRON)),\n",
    "                 (\"B3_drop_user\", lambda t: remove_stopwords(t, SW_NOUSER)),\n",
    "                 (\"C_stop+lemma\", lambda t: lemmatize_text(remove_stopwords(t))),\n",
    "                 (\"C2_lemma+pron\", lambda t: lemmatize_text(remove_stopwords(t, SW_PRON)))]:\n",
    "    tr_text = train.text_clean.map(fn)\n",
    "    va_text = val.text_clean.map(fn)\n",
    "    stats_rows.append({\n",
    "        \"variant\": name,\n",
    "        \"avg_words_train\": round(tr_text.str.split().str.len().mean(), 2),\n",
    "        \"vocab_train\": len({w for t in tr_text for w in t.split()}),\n",
    "        \"empty_train\": int((tr_text.str.strip() == \"\").sum()),\n",
    "        \"empty_val\": int((va_text.str.strip() == \"\").sum()),\n",
    "    })\n",
    "cost_table = pd.DataFrame(stats_rows).set_index(\"variant\")\n",
    "cost_table"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "be7933af",
   "metadata": {},
   "source": [
    "### Reading the cost table\n",
    "\n",
    "* **empty tweets** (variant B3 only): TF-IDF handles them as an all-zero row — they are simply not\n",
    "  informative any more. They are counted and reported, never deleted (deleting them would change\n",
    "  the class distribution and the train/validation comparability).\n",
    "* **vocabulary size** drops sharply with lemmatisation, which is exactly the point: fewer, denser\n",
    "  features for a small dataset."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "6cacfa6e",
   "metadata": {},
   "source": [
    "## 4 · The experiment (identical conditions for every variant)\n",
    "\n",
    "* TF-IDF fitted on **train only**, validation only `transform`-ed (no leakage);\n",
    "* `token_pattern=r\"[a-z0-9'*]+\"` **everywhere** — the default tokenizer splits `don't` into\n",
    "  `don` + `t` and drops `f***` (single-letter token), which would undo Emna's cleaning *and* the\n",
    "  protected negations;\n",
    "* Logistic Regression (`max_iter=1000`, `random_state=42`), target = validation **F1 of class 1**."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 10,
   "id": "294ef8f9",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:12.836729Z",
     "iopub.status.busy": "2026-11-10T18:55:12.836530Z",
     "iopub.status.idle": "2026-11-10T18:55:12.842840Z",
     "shell.execute_reply": "2026-11-10T18:55:12.842202Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/plain": [
       "{'max_features': 20000,\n",
       " 'ngram_range': (1, 2),\n",
       " 'min_df': 2,\n",
       " 'token_pattern': \"[a-z0-9'*]+\"}"
      ]
     },
     "execution_count": 10,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "from sklearn.feature_extraction.text import TfidfVectorizer\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from sklearn.metrics import confusion_matrix\n",
    "from src.evaluation import calculate_metrics          # accuracy, precision, recall, f1\n",
    "\n",
    "TFIDF_PARAMS = dict(max_features=20000, ngram_range=(1, 2), min_df=2,\n",
    "                    token_pattern=r\"[a-z0-9'*]+\")     # keeps don't and f***\n",
    "\n",
    "VARIANTS = {\n",
    "    \"A_clean_only\":           (lambda t: t,                                    None),\n",
    "    \"B_stopwords\":            (remove_stopwords,                               SW_DEFAULT),\n",
    "    \"B2_keep_pron\":           (lambda t: remove_stopwords(t, SW_PRON),          SW_PRON),\n",
    "    \"B3_drop_user\":           (lambda t: remove_stopwords(t, SW_NOUSER),        SW_NOUSER),\n",
    "    \"C_stop+lemma\":           (lambda t: lemmatize_text(remove_stopwords(t)),   SW_DEFAULT),\n",
    "    \"C2_lemma+pron\":          (lambda t: lemmatize_text(remove_stopwords(t, SW_PRON)), SW_PRON),\n",
    "}\n",
    "TFIDF_PARAMS"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 11,
   "id": "ef0ae6b5",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:12.844374Z",
     "iopub.status.busy": "2026-11-10T18:55:12.844213Z",
     "iopub.status.idle": "2026-11-10T18:55:22.428668Z",
     "shell.execute_reply": "2026-11-10T18:55:22.428056Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "A_clean_only     F1=0.4777  accuracy=0.7440  (1.6s)\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "B_stopwords      F1=0.4871  accuracy=0.7455  (1.3s)\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "B2_keep_pron     F1=0.5030  accuracy=0.7523  (1.5s)\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "B3_drop_user     F1=0.4992  accuracy=0.7485  (0.9s)\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "C_stop+lemma     F1=0.5205  accuracy=0.7523  (2.1s)\n"
     ]
    },
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "C2_lemma+pron    F1=0.5319  accuracy=0.7621  (2.1s)\n"
     ]
    }
   ],
   "source": [
    "def run_variant(name, fn):\n",
    "    \"\"\"Train + evaluate ONE text transformation. Everything else is fixed.\"\"\"\n",
    "    t0 = time.time()\n",
    "    X_train_text = train[\"text_clean\"].map(fn)\n",
    "    X_val_text   = val[\"text_clean\"].map(fn)\n",
    "\n",
    "    vectorizer = TfidfVectorizer(**TFIDF_PARAMS)\n",
    "    X_train = vectorizer.fit_transform(X_train_text)      # fit on TRAIN only\n",
    "    X_val   = vectorizer.transform(X_val_text)            # transform only\n",
    "\n",
    "    model = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, train[\"label\"])\n",
    "    pred  = model.predict(X_val)\n",
    "\n",
    "    row = {\n",
    "        \"variant\": name,\n",
    "        \"n_features\": int(X_train.shape[1]),\n",
    "        \"empty_train\": int((X_train_text.str.strip() == \"\").sum()),\n",
    "        \"empty_val\": int((X_val_text.str.strip() == \"\").sum()),\n",
    "        **calculate_metrics(val[\"label\"], pred),\n",
    "        \"f1_macro\": float(__import__(\"sklearn.metrics\", fromlist=[\"f1_score\"])\n",
    "                          .f1_score(val[\"label\"], pred, average=\"macro\")),\n",
    "        \"seconds\": round(time.time() - t0, 1),\n",
    "    }\n",
    "    return row, confusion_matrix(val[\"label\"], pred, labels=[0, 1]), pred\n",
    "\n",
    "\n",
    "rows, matrices, predictions = [], {}, {}\n",
    "for name, (fn, sw) in VARIANTS.items():\n",
    "    row, cm, pred = run_variant(name, fn)\n",
    "    rows.append(row); matrices[name] = cm; predictions[name] = pred\n",
    "    print(f\"{name:<16} F1={row['f1']:.4f}  accuracy={row['accuracy']:.4f}  ({row['seconds']}s)\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 12,
   "id": "c4410d2c",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:22.430496Z",
     "iopub.status.busy": "2026-11-10T18:55:22.430385Z",
     "iopub.status.idle": "2026-11-10T18:55:22.445930Z",
     "shell.execute_reply": "2026-11-10T18:55:22.445425Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>n_features</th>\n",
       "      <th>empty_train</th>\n",
       "      <th>empty_val</th>\n",
       "      <th>accuracy</th>\n",
       "      <th>precision</th>\n",
       "      <th>recall</th>\n",
       "      <th>f1</th>\n",
       "      <th>f1_macro</th>\n",
       "      <th>seconds</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>variant</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>C2_lemma+pron</th>\n",
       "      <td>20000</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>0.7621</td>\n",
       "      <td>0.8364</td>\n",
       "      <td>0.3900</td>\n",
       "      <td>0.5319</td>\n",
       "      <td>0.6862</td>\n",
       "      <td>2.1</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>C_stop+lemma</th>\n",
       "      <td>17559</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>0.7523</td>\n",
       "      <td>0.7911</td>\n",
       "      <td>0.3878</td>\n",
       "      <td>0.5205</td>\n",
       "      <td>0.6767</td>\n",
       "      <td>2.1</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B2_keep_pron</th>\n",
       "      <td>20000</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>0.7523</td>\n",
       "      <td>0.8259</td>\n",
       "      <td>0.3617</td>\n",
       "      <td>0.5030</td>\n",
       "      <td>0.6690</td>\n",
       "      <td>1.5</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B3_drop_user</th>\n",
       "      <td>16379</td>\n",
       "      <td>58</td>\n",
       "      <td>6</td>\n",
       "      <td>0.7485</td>\n",
       "      <td>0.8058</td>\n",
       "      <td>0.3617</td>\n",
       "      <td>0.4992</td>\n",
       "      <td>0.6657</td>\n",
       "      <td>0.9</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B_stopwords</th>\n",
       "      <td>17644</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>0.7455</td>\n",
       "      <td>0.8081</td>\n",
       "      <td>0.3486</td>\n",
       "      <td>0.4871</td>\n",
       "      <td>0.6589</td>\n",
       "      <td>1.3</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>A_clean_only</th>\n",
       "      <td>20000</td>\n",
       "      <td>0</td>\n",
       "      <td>0</td>\n",
       "      <td>0.7440</td>\n",
       "      <td>0.8158</td>\n",
       "      <td>0.3377</td>\n",
       "      <td>0.4777</td>\n",
       "      <td>0.6540</td>\n",
       "      <td>1.6</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "               n_features  empty_train  empty_val  accuracy  precision  \\\n",
       "variant                                                                  \n",
       "C2_lemma+pron       20000            0          0    0.7621     0.8364   \n",
       "C_stop+lemma        17559            0          0    0.7523     0.7911   \n",
       "B2_keep_pron        20000            0          0    0.7523     0.8259   \n",
       "B3_drop_user        16379           58          6    0.7485     0.8058   \n",
       "B_stopwords         17644            0          0    0.7455     0.8081   \n",
       "A_clean_only        20000            0          0    0.7440     0.8158   \n",
       "\n",
       "               recall      f1  f1_macro  seconds  \n",
       "variant                                           \n",
       "C2_lemma+pron  0.3900  0.5319    0.6862      2.1  \n",
       "C_stop+lemma   0.3878  0.5205    0.6767      2.1  \n",
       "B2_keep_pron   0.3617  0.5030    0.6690      1.5  \n",
       "B3_drop_user   0.3617  0.4992    0.6657      0.9  \n",
       "B_stopwords    0.3486  0.4871    0.6589      1.3  \n",
       "A_clean_only   0.3377  0.4777    0.6540      1.6  "
      ]
     },
     "execution_count": 12,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "comparison = (pd.DataFrame(rows).set_index(\"variant\")\n",
    "                .sort_values(\"f1\", ascending=False))\n",
    "comparison.round(4)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 13,
   "id": "570b3247",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:22.447645Z",
     "iopub.status.busy": "2026-11-10T18:55:22.447541Z",
     "iopub.status.idle": "2026-11-10T18:55:22.453338Z",
     "shell.execute_reply": "2026-11-10T18:55:22.452751Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "saved: results/metrics/03_preprocessing_experiments/preprocessing_comparison.csv\n",
      "A_clean_only : accuracy=0.744 precision=0.816 recall=0.338 F1=0.478\n",
      "guide        : accuracy≈0.740 precision≈0.820 recall≈0.330 F1≈0.470\n",
      "\n",
      "✔ setup matches the guide: the comparison below is trustworthy\n"
     ]
    }
   ],
   "source": [
    "# --- save the metric table (deliverable) ----------------------------------\n",
    "comparison.round(4).to_csv(METRIC_DIR / \"preprocessing_comparison.csv\")\n",
    "print(\"saved:\", (METRIC_DIR / \"preprocessing_comparison.csv\").relative_to(PROJECT_ROOT))\n",
    "\n",
    "# --- reproduce the reference point of the guide (section 2) ---------------\n",
    "base = comparison.loc[\"A_clean_only\"]\n",
    "print(f\"A_clean_only : accuracy={base.accuracy:.3f} precision={base.precision:.3f} \"\n",
    "      f\"recall={base.recall:.3f} F1={base.f1:.3f}\")\n",
    "print(\"guide        : accuracy≈0.740 precision≈0.820 recall≈0.330 F1≈0.470\")\n",
    "assert 0.72 < base.accuracy < 0.76 and 0.44 < base.f1 < 0.51, \"setup differs from the guide - check the cleaning/token_pattern\"\n",
    "print(\"\\n✔ setup matches the guide: the comparison below is trustworthy\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d3c2051d",
   "metadata": {},
   "source": [
    "## 5 · Figures\n",
    "\n",
    "All figures are saved in `results/figures/03_preprocessing_experiments/`."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 14,
   "id": "3668a1a5",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:22.454973Z",
     "iopub.status.busy": "2026-11-10T18:55:22.454865Z",
     "iopub.status.idle": "2026-11-10T18:55:22.786064Z",
     "shell.execute_reply": "2026-11-10T18:55:22.785783Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 990x495 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# --- figure 1: F1 of the offensive class per variant -----------------------\n",
    "fig, ax = plt.subplots(figsize=(9, 4.5))\n",
    "ordered = comparison.sort_values(\"f1\")\n",
    "colors = [\"#c44e52\" if v == \"A_clean_only\" else \"#4c72b0\" for v in ordered.index]\n",
    "bars = ax.barh(ordered.index, ordered[\"f1\"], color=colors)\n",
    "ax.axvline(base.f1, color=\"#c44e52\", ls=\"--\", lw=1,\n",
    "           label=f\"reference A = {base.f1:.3f}\")\n",
    "for bar, value in zip(bars, ordered[\"f1\"]):\n",
    "    ax.text(value + 0.004, bar.get_y() + bar.get_height() / 2, f\"{value:.3f}\",\n",
    "            va=\"center\", fontsize=9)\n",
    "ax.set(xlabel=\"Validation F1 — offensive class\", ylabel=\"\",\n",
    "       title=\"Notebook 03 — effect of stopwords & lemmatisation (same TF-IDF, same model)\")\n",
    "ax.set_xlim(0, max(ordered[\"f1\"]) + 0.06)\n",
    "ax.legend(loc=\"upper center\", bbox_to_anchor=(0.5, -0.16), frameon=False)   # below the axis, no overlap\n",
    "plt.tight_layout()\n",
    "plt.savefig(FIGURE_DIR / \"f1_by_variant.png\", dpi=150, bbox_inches=\"tight\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 15,
   "id": "e3245997",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:22.787519Z",
     "iopub.status.busy": "2026-11-10T18:55:22.787382Z",
     "iopub.status.idle": "2026-11-10T18:55:23.155380Z",
     "shell.execute_reply": "2026-11-10T18:55:23.154931Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1100x495 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# --- figure 2: the four metrics side by side ------------------------------\n",
    "metric_names = [\"accuracy\", \"precision\", \"recall\", \"f1\"]\n",
    "plot_df = comparison.loc[list(VARIANTS)][metric_names]\n",
    "axis = plot_df.plot(kind=\"bar\", figsize=(10, 4.5), rot=20,\n",
    "                    color=[\"#8c8c8c\", \"#55a868\", \"#c44e52\", \"#4c72b0\"])\n",
    "axis.axhline(1 - val.label.mean(), color=\"k\", ls=\":\", lw=1,\n",
    "             label=f\"majority baseline accuracy = {1 - val.label.mean():.3f}\")\n",
    "axis.set(ylabel=\"score (validation)\", xlabel=\"\",\n",
    "         title=\"Accuracy is misleading here: look at recall and F1\")\n",
    "axis.legend(ncol=5, fontsize=8, loc=\"upper center\", bbox_to_anchor=(0.5, -0.22))\n",
    "plt.tight_layout()\n",
    "plt.savefig(FIGURE_DIR / \"metrics_by_variant.png\", dpi=150, bbox_inches=\"tight\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 16,
   "id": "77c802ed",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:23.157055Z",
     "iopub.status.busy": "2026-11-10T18:55:23.156909Z",
     "iopub.status.idle": "2026-11-10T18:55:24.402005Z",
     "shell.execute_reply": "2026-11-10T18:55:24.401578Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1320x770 with 6 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# --- figure 3: one confusion matrix per variant ---------------------------\n",
    "fig, axes = plt.subplots(2, 3, figsize=(12, 7))\n",
    "for ax, (name, cm) in zip(axes.ravel(), matrices.items()):\n",
    "    sns.heatmap(cm, annot=True, fmt=\"d\", cmap=\"Blues\", cbar=False, ax=ax,\n",
    "                xticklabels=[\"pred 0\", \"pred 1\"], yticklabels=[\"true 0\", \"true 1\"])\n",
    "    ax.set_title(f\"{name}\\nF1={comparison.loc[name, 'f1']:.3f}  \"\n",
    "                 f\"(missed offensive: {cm[1, 0]}, false alarms: {cm[0, 1]})\", fontsize=9)\n",
    "    ax.set(xlabel=\"prediction\", ylabel=\"truth\")\n",
    "fig.suptitle(\"Validation confusion matrices — the low recall problem is visible in every variant\",\n",
    "             fontsize=12)\n",
    "plt.tight_layout()\n",
    "plt.savefig(FIGURE_DIR / \"confusion_matrices.png\", dpi=150, bbox_inches=\"tight\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 17,
   "id": "ec9a9775",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:24.403606Z",
     "iopub.status.busy": "2026-11-10T18:55:24.403411Z",
     "iopub.status.idle": "2026-11-10T18:55:24.830600Z",
     "shell.execute_reply": "2026-11-10T18:55:24.830235Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1210x440 with 2 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# --- figure 4: what the transformations cost ------------------------------\n",
    "fig, axes = plt.subplots(1, 2, figsize=(11, 4))\n",
    "cost_table[\"vocab_train\"].plot(kind=\"bar\", ax=axes[0], rot=25, color=\"#4c72b0\")\n",
    "axes[0].set(title=\"Vocabulary size on the train set\", ylabel=\"distinct tokens\", xlabel=\"\")\n",
    "cost_table[\"avg_words_train\"].plot(kind=\"bar\", ax=axes[1], rot=25, color=\"#55a868\")\n",
    "axes[1].set(title=\"Average words per tweet\", ylabel=\"words\", xlabel=\"\")\n",
    "fig.suptitle(f\"Cost of each transformation (B3 leaves {cost_table.loc['B3_drop_user','empty_train']} \"\n",
    "             f\"training and {cost_table.loc['B3_drop_user','empty_val']} validation tweets empty)\", fontsize=11)\n",
    "plt.tight_layout()\n",
    "plt.savefig(FIGURE_DIR / \"vocabulary_and_cost.png\", dpi=150, bbox_inches=\"tight\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "8491b50c",
   "metadata": {},
   "source": [
    "## 6 · Robustness check — separating a real effect from validation noise\n",
    "\n",
    "The validation set has only **1,324 tweets**: a 1-point F1 difference can be pure noise (guide,\n",
    "pitfall §7). So every variant is re-scored with a **stratified 5-fold cross-validation on the\n",
    "training set** — same folds for all variants (paired comparison), vectorizer fitted **inside** each\n",
    "fold (no leakage). The validation set is *not* part of it, so it stays an independent judge.\n",
    "\n",
    "Two estimates of the same quantity are expected to differ slightly (each CV model sees 80 % of the\n",
    "data, so it is usually ~2 points more pessimistic). What matters is whether the **ranking** and the\n",
    "**sign of the paired differences** agree."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 18,
   "id": "be9c3800",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:55:24.832476Z",
     "iopub.status.busy": "2026-11-10T18:55:24.832337Z",
     "iopub.status.idle": "2026-11-10T18:56:07.475620Z",
     "shell.execute_reply": "2026-11-10T18:56:07.475215Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>cv_f1_mean</th>\n",
       "      <th>cv_f1_std</th>\n",
       "      <th>validation_f1</th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>C_stop+lemma</th>\n",
       "      <td>0.5013</td>\n",
       "      <td>0.0133</td>\n",
       "      <td>0.5205</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B3_drop_user</th>\n",
       "      <td>0.4855</td>\n",
       "      <td>0.0074</td>\n",
       "      <td>0.4992</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>C2_lemma+pron</th>\n",
       "      <td>0.4850</td>\n",
       "      <td>0.0124</td>\n",
       "      <td>0.5319</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B_stopwords</th>\n",
       "      <td>0.4758</td>\n",
       "      <td>0.0129</td>\n",
       "      <td>0.4871</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>B2_keep_pron</th>\n",
       "      <td>0.4605</td>\n",
       "      <td>0.0093</td>\n",
       "      <td>0.5030</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>A_clean_only</th>\n",
       "      <td>0.4570</td>\n",
       "      <td>0.0239</td>\n",
       "      <td>0.4777</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "               cv_f1_mean  cv_f1_std  validation_f1\n",
       "C_stop+lemma       0.5013     0.0133         0.5205\n",
       "B3_drop_user       0.4855     0.0074         0.4992\n",
       "C2_lemma+pron      0.4850     0.0124         0.5319\n",
       "B_stopwords        0.4758     0.0129         0.4871\n",
       "B2_keep_pron       0.4605     0.0093         0.5030\n",
       "A_clean_only       0.4570     0.0239         0.4777"
      ]
     },
     "execution_count": 18,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "from sklearn.model_selection import StratifiedKFold\n",
    "from sklearn.pipeline import Pipeline\n",
    "from sklearn.metrics import f1_score\n",
    "\n",
    "skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)\n",
    "FOLDS = list(skf.split(np.zeros(len(train)), train[\"label\"]))   # identical folds for every variant\n",
    "\n",
    "cv_scores = {}\n",
    "for name, (fn, _) in VARIANTS.items():\n",
    "    texts = train[\"text_clean\"].map(fn)\n",
    "    scores = []\n",
    "    for fold_id, (idx_tr, idx_te) in enumerate(FOLDS, start=1):\n",
    "        pipe = Pipeline([(\"tfidf\", TfidfVectorizer(**TFIDF_PARAMS)),\n",
    "                         (\"clf\", LogisticRegression(max_iter=1000, random_state=42))])\n",
    "        pipe.fit(texts.iloc[idx_tr], train[\"label\"].iloc[idx_tr])\n",
    "        scores.append(f1_score(train[\"label\"].iloc[idx_te], pipe.predict(texts.iloc[idx_te]),\n",
    "                               pos_label=1))\n",
    "    cv_scores[name] = np.array(scores)\n",
    "\n",
    "cv_folds = pd.DataFrame(cv_scores, index=[f\"fold {i}\" for i in range(1, 6)])\n",
    "cv_table = cv_folds.agg([\"mean\", \"std\"]).T.rename(columns={\"mean\": \"cv_f1_mean\", \"std\": \"cv_f1_std\"})\n",
    "cv_table[\"validation_f1\"] = comparison[\"f1\"]\n",
    "cv_table = cv_table[[\"cv_f1_mean\", \"cv_f1_std\", \"validation_f1\"]].sort_values(\"cv_f1_mean\", ascending=False)\n",
    "cv_table.round(4)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 19,
   "id": "946562e9",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:56:07.477971Z",
     "iopub.status.busy": "2026-11-10T18:56:07.477407Z",
     "iopub.status.idle": "2026-11-10T18:56:07.497841Z",
     "shell.execute_reply": "2026-11-10T18:56:07.497582Z"
    }
   },
   "outputs": [
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>cv_gain_mean_pt</th>\n",
       "      <th>cv_gain_std_pt</th>\n",
       "      <th>validation_gain_pt</th>\n",
       "      <th>same sign on both estimates?</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>comparison</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>stopwords (B - A)</th>\n",
       "      <td>1.88</td>\n",
       "      <td>1.88</td>\n",
       "      <td>0.94</td>\n",
       "      <td>yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>pronouns (B2 - B)</th>\n",
       "      <td>-1.52</td>\n",
       "      <td>0.96</td>\n",
       "      <td>1.60</td>\n",
       "      <td>NO  ← tie / noise</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>pronouns (C2 - C)</th>\n",
       "      <td>-1.63</td>\n",
       "      <td>1.14</td>\n",
       "      <td>1.15</td>\n",
       "      <td>NO  ← tie / noise</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>drop 'user' (B3 - B)</th>\n",
       "      <td>0.98</td>\n",
       "      <td>1.00</td>\n",
       "      <td>1.22</td>\n",
       "      <td>yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>lemmatise (C - B)</th>\n",
       "      <td>2.55</td>\n",
       "      <td>1.86</td>\n",
       "      <td>3.34</td>\n",
       "      <td>yes</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>lemmatise (C2 - B2)</th>\n",
       "      <td>2.45</td>\n",
       "      <td>0.71</td>\n",
       "      <td>2.89</td>\n",
       "      <td>yes</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                      cv_gain_mean_pt  cv_gain_std_pt  validation_gain_pt  \\\n",
       "comparison                                                                  \n",
       "stopwords (B - A)                1.88            1.88                0.94   \n",
       "pronouns (B2 - B)               -1.52            0.96                1.60   \n",
       "pronouns (C2 - C)               -1.63            1.14                1.15   \n",
       "drop 'user' (B3 - B)             0.98            1.00                1.22   \n",
       "lemmatise (C - B)                2.55            1.86                3.34   \n",
       "lemmatise (C2 - B2)              2.45            0.71                2.89   \n",
       "\n",
       "                     same sign on both estimates?  \n",
       "comparison                                         \n",
       "stopwords (B - A)                             yes  \n",
       "pronouns (B2 - B)               NO  ← tie / noise  \n",
       "pronouns (C2 - C)               NO  ← tie / noise  \n",
       "drop 'user' (B3 - B)                          yes  \n",
       "lemmatise (C - B)                             yes  \n",
       "lemmatise (C2 - B2)                           yes  "
      ]
     },
     "execution_count": 19,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "# --- paired differences: same folds, so the noise partly cancels -----------\n",
    "PAIRS = {\"stopwords (B - A)\": (\"B_stopwords\", \"A_clean_only\"),\n",
    "         \"pronouns (B2 - B)\": (\"B2_keep_pron\", \"B_stopwords\"),\n",
    "         \"pronouns (C2 - C)\": (\"C2_lemma+pron\", \"C_stop+lemma\"),\n",
    "         \"drop 'user' (B3 - B)\": (\"B3_drop_user\", \"B_stopwords\"),\n",
    "         \"lemmatise (C - B)\": (\"C_stop+lemma\", \"B_stopwords\"),\n",
    "         \"lemmatise (C2 - B2)\": (\"C2_lemma+pron\", \"B2_keep_pron\")}\n",
    "\n",
    "paired = pd.DataFrame([{\n",
    "    \"comparison\": label,\n",
    "    \"cv_gain_mean_pt\": round(100 * (cv_folds[hi] - cv_folds[lo]).mean(), 2),\n",
    "    \"cv_gain_std_pt\": round(100 * (cv_folds[hi] - cv_folds[lo]).std(), 2),\n",
    "    \"validation_gain_pt\": round(100 * (comparison.loc[hi, \"f1\"] - comparison.loc[lo, \"f1\"]), 2),\n",
    "} for label, (hi, lo) in PAIRS.items()]).set_index(\"comparison\")\n",
    "paired[\"same sign on both estimates?\"] = np.where(\n",
    "    np.sign(paired.cv_gain_mean_pt) == np.sign(paired.validation_gain_pt), \"yes\", \"NO  ← tie / noise\")\n",
    "\n",
    "cv_folds.round(4).to_csv(METRIC_DIR / \"cross_validation_check.csv\")\n",
    "paired.round(2)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 20,
   "id": "bcc2f6d4",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:56:07.499385Z",
     "iopub.status.busy": "2026-11-10T18:56:07.499273Z",
     "iopub.status.idle": "2026-11-10T18:56:07.850965Z",
     "shell.execute_reply": "2026-11-10T18:56:07.850346Z"
    }
   },
   "outputs": [
    {
     "data": {
      "image/png": "...",
      "text/plain": [
       "<Figure size 1100x484 with 1 Axes>"
      ]
     },
     "metadata": {},
     "output_type": "display_data"
    }
   ],
   "source": [
    "# --- figure 5: five folds per variant ------------------------------------\n",
    "fig, ax = plt.subplots(figsize=(10, 4.4))\n",
    "order = cv_table.index\n",
    "ax.boxplot([cv_folds[name] for name in order], tick_labels=order, widths=0.5,\n",
    "           medianprops=dict(color=\"#c44e52\"))\n",
    "ax.plot(range(1, len(order) + 1), comparison.loc[order, \"f1\"], \"o\", color=\"#4c72b0\",\n",
    "        label=\"validation F1\", zorder=3)\n",
    "ax.set(ylabel=\"F1 — offensive class\",\n",
    "       title=\"5-fold CV on train (box: spread over folds) vs validation F1 (dots)\")\n",
    "ax.legend(loc=\"lower right\")\n",
    "plt.tight_layout()\n",
    "plt.savefig(FIGURE_DIR / \"cross_validation_check.png\", dpi=150, bbox_inches=\"tight\")\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "46a57743",
   "metadata": {},
   "source": [
    "### What the two estimates say\n",
    "\n",
    "* **Lemmatisation is the real effect.** It is the only transformation whose gain is large\n",
    "  (**+4 pt** over B, in both estimates) and whose sign never flips. It also removes ~20 % of the\n",
    "  vocabulary. Keep it.\n",
    "* **Stopword removal also helps** (+0.9 pt validation, +1.9 pt in CV, same sign): keep it.\n",
    "* **The `user` placeholder and the pronouns are the two ambiguous cases.** Dropping `user` gives a\n",
    "  small gain in both estimates, but it *empties 58 training tweets*: the gain is smaller than the\n",
    "  fold-to-fold spread, so it is not worth losing data.\n",
    "* **Pronouns are a genuine contradiction**: validation says *keep* them (+1.6 pt on B, +1.1 pt on\n",
    "  C), the 5-fold CV says *drop* them (−1.5 pt, −1.6 pt). The differences are the same order of\n",
    "  magnitude as the CV standard deviation (~1.2 pt). This is exactly the pitfall the guide warns\n",
    "  about on a 1,324-tweet validation set, and the honest conclusion is: **the pronoun effect is not\n",
    "  established**.\n",
    "\n",
    "We therefore apply a rule **declared before looking at the winner**, so that the decision cannot be\n",
    "tuned after the fact: *when the validation set and the cross-validation disagree in sign, take the\n",
    "variant with the better CV mean (it is estimated on 5 × 9,533 tweets instead of 1,324) and, on a\n",
    "tie, the one with the smaller vocabulary.* This gives variant **C = `stopwords (negations protected)\n",
    "+ lemmatisation`**, and the runner-up **C2** (same thing but pronouns kept) is carried into notebook\n",
    "04 as a sensitivity check — if the balancing conclusion depends on that 1-point choice, we will\n",
    "see it there."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f8fb391b",
   "metadata": {},
   "source": [
    "## 7 · Decision\n",
    "\n",
    "The four questions of the guide, answered with the two estimates. The table is computed\n",
    "**automatically from the results** (no hand-typed score), so it cannot drift from the experiment."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 21,
   "id": "40072444",
   "metadata": {
    "execution": {
     "iopub.execute_input": "2026-11-10T18:56:07.852876Z",
     "iopub.status.busy": "2026-11-10T18:56:07.852733Z",
     "iopub.status.idle": "2026-11-10T18:56:07.867514Z",
     "shell.execute_reply": "2026-11-10T18:56:07.866895Z"
    }
   },
   "outputs": [
    {
     "name": "stdout",
     "output_type": "stream",
     "text": [
      "best on validation : C2_lemma+pron (0.5319)\n",
      "best on 5-fold CV  : C_stop+lemma (0.5013)\n",
      "kept for notebook 04: C_stop+lemma | runner-up: C2_lemma+pron\n"
     ]
    },
    {
     "data": {
      "text/html": [
       "<div>\n",
       "<style scoped>\n",
       "    .dataframe tbody tr th:only-of-type {\n",
       "        vertical-align: middle;\n",
       "    }\n",
       "\n",
       "    .dataframe tbody tr th {\n",
       "        vertical-align: top;\n",
       "    }\n",
       "\n",
       "    .dataframe thead th {\n",
       "        text-align: right;\n",
       "    }\n",
       "</style>\n",
       "<table border=\"1\" class=\"dataframe\">\n",
       "  <thead>\n",
       "    <tr style=\"text-align: right;\">\n",
       "      <th></th>\n",
       "      <th>Decision</th>\n",
       "      <th>Evidence</th>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Question</th>\n",
       "      <th></th>\n",
       "      <th></th>\n",
       "    </tr>\n",
       "  </thead>\n",
       "  <tbody>\n",
       "    <tr>\n",
       "      <th>Remove stopwords (negations protected)?</th>\n",
       "      <td>YES</td>\n",
       "      <td>0.487 vs 0.478 (val +0.9 pt) and 0.476 vs 0.457 (CV +1.9 pt): sa...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Keep pronouns?</th>\n",
       "      <td>NO — not established (drop pronouns)</td>\n",
       "      <td>without lemma: val +1.6 pt / CV -1.5 pt; with lemma: val +1.1 pt...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Drop the 'user' placeholder?</th>\n",
       "      <td>NO</td>\n",
       "      <td>val +1.2 pt / CV +1.0 pt, but it empties 58 training and 6 valid...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>Lemmatise?</th>\n",
       "      <td>YES</td>\n",
       "      <td>0.520 vs 0.487 (val +3.3 pt) and 0.501 vs 0.476 (CV +2.6 pt); vo...</td>\n",
       "    </tr>\n",
       "    <tr>\n",
       "      <th>→ TEXT PIPELINE KEPT FOR NOTEBOOK 04</th>\n",
       "      <td>C_stop+lemma</td>\n",
       "      <td>F1 0.520 on validation (rank 2), 0.501 in 5-fold CV (rank 1); ru...</td>\n",
       "    </tr>\n",
       "  </tbody>\n",
       "</table>\n",
       "</div>"
      ],
      "text/plain": [
       "                                                                     Decision  \\\n",
       "Question                                                                        \n",
       "Remove stopwords (negations protected)?                                   YES   \n",
       "Keep pronouns?                           NO — not established (drop pronouns)   \n",
       "Drop the 'user' placeholder?                                               NO   \n",
       "Lemmatise?                                                                YES   \n",
       "→ TEXT PIPELINE KEPT FOR NOTEBOOK 04                             C_stop+lemma   \n",
       "\n",
       "                                                                                                    Evidence  \n",
       "Question                                                                                                      \n",
       "Remove stopwords (negations protected)?  0.487 vs 0.478 (val +0.9 pt) and 0.476 vs 0.457 (CV +1.9 pt): sa...  \n",
       "Keep pronouns?                           without lemma: val +1.6 pt / CV -1.5 pt; with lemma: val +1.1 pt...  \n",
       "Drop the 'user' placeholder?             val +1.2 pt / CV +1.0 pt, but it empties 58 training and 6 valid...  \n",
       "Lemmatise?                               0.520 vs 0.487 (val +3.3 pt) and 0.501 vs 0.476 (CV +2.6 pt); vo...  \n",
       "→ TEXT PIPELINE KEPT FOR NOTEBOOK 04     F1 0.520 on validation (rank 2), 0.501 in 5-fold CV (rank 1); ru...  "
      ]
     },
     "execution_count": 21,
     "metadata": {},
     "output_type": "execute_result"
    }
   ],
   "source": [
    "F1  = comparison[\"f1\"]\n",
    "CV  = cv_table[\"cv_f1_mean\"]\n",
    "MATERIAL = 0.01        # 1 F1 point: below that, with 1,324 validation tweets, it is noise\n",
    "\n",
    "def both_agree(hi, lo):\n",
    "    # validation gain and cross-validation gain between two variants\n",
    "    return F1[hi] - F1[lo], CV[hi] - CV[lo]\n",
    "\n",
    "rows = []\n",
    "\n",
    "# 1. stopwords: accepted if the two estimates agree and the gain is positive\n",
    "val_gain, cv_gain = both_agree(\"B_stopwords\", \"A_clean_only\")\n",
    "rows.append((\"Remove stopwords (negations protected)?\",\n",
    "             \"YES\" if val_gain > 0 and cv_gain > 0 else \"NO\",\n",
    "             f\"{F1['B_stopwords']:.3f} vs {F1['A_clean_only']:.3f} (val {100*val_gain:+.1f} pt) and \"\n",
    "             f\"{CV['B_stopwords']:.3f} vs {CV['A_clean_only']:.3f} (CV {100*cv_gain:+.1f} pt): same sign \"\n",
    "             f\"in both estimates; the gain is small but free (no data lost, negations preserved)\"))\n",
    "\n",
    "# 2. pronouns: accepted only if the two estimates point the same way\n",
    "val_gain, cv_gain = both_agree(\"C2_lemma+pron\", \"C_stop+lemma\")\n",
    "val_gain2, cv_gain2 = both_agree(\"B2_keep_pron\", \"B_stopwords\")\n",
    "if np.sign(val_gain) == np.sign(cv_gain):\n",
    "    pron_decision = \"KEEP\" if val_gain > 0 else \"DROP\"\n",
    "else:\n",
    "    pron_decision = \"NO — not established (drop pronouns)\"\n",
    "rows.append((\"Keep pronouns?\",\n",
    "             pron_decision,\n",
    "             f\"without lemma: val {100*val_gain2:+.1f} pt / CV {100*cv_gain2:+.1f} pt; \"\n",
    "             f\"with lemma: val {100*val_gain:+.1f} pt / CV {100*cv_gain:+.1f} pt. \"\n",
    "             f\"The two estimates contradict each other and the effect is below the CV spread \"\n",
    "             f\"(±{100*cv_table.loc['C_stop+lemma','cv_f1_std']:.1f} pt) → noise; we keep the smaller vocabulary\"))\n",
    "\n",
    "# 3. 'user' placeholder: refused because it destroys tweets\n",
    "val_gain, cv_gain = both_agree(\"B3_drop_user\", \"B_stopwords\")\n",
    "empty_lost = int(cost_table.loc[\"B3_drop_user\", \"empty_train\"])\n",
    "rows.append((\"Drop the 'user' placeholder?\",\n",
    "             \"NO\" if empty_lost > 0 or min(val_gain, cv_gain) <= MATERIAL else \"YES\",\n",
    "             f\"val {100*val_gain:+.1f} pt / CV {100*cv_gain:+.1f} pt, but it empties {empty_lost} training \"\n",
    "             f\"and {int(cost_table.loc['B3_drop_user','empty_val'])} validation tweets: the gain is inside \"\n",
    "             f\"the noise, the data loss is real → refused\"))\n",
    "\n",
    "# 4. lemmatisation: the only effect that is both large and consistent\n",
    "val_gain, cv_gain = both_agree(\"C_stop+lemma\", \"B_stopwords\")\n",
    "rows.append((\"Lemmatise?\",\n",
    "             \"YES\" if (val_gain >= MATERIAL and cv_gain > 0) else \"NO\",\n",
    "             f\"{F1['C_stop+lemma']:.3f} vs {F1['B_stopwords']:.3f} (val {100*val_gain:+.1f} pt) and \"\n",
    "             f\"{CV['C_stop+lemma']:.3f} vs {CV['B_stopwords']:.3f} (CV {100*cv_gain:+.1f} pt); vocabulary \"\n",
    "             f\"{cost_table.loc['B_stopwords','vocab_train']} → {cost_table.loc['C_stop+lemma','vocab_train']} tokens; \"\n",
    "             f\"cost {comparison.loc['C_stop+lemma','seconds']:.1f}s vs {comparison.loc['B_stopwords','seconds']:.1f}s\"))\n",
    "\n",
    "# 5. the retained pipeline (rule declared in section 6: CV wins when the signs disagree)\n",
    "best_variant = cv_table.index[0]\n",
    "runner_up    = comparison.index[0]\n",
    "rows.append((\"→ TEXT PIPELINE KEPT FOR NOTEBOOK 04\",\n",
    "             best_variant,\n",
    "             f\"F1 {F1[best_variant]:.3f} on validation (rank {list(comparison.index).index(best_variant)+1}), \"\n",
    "             f\"{CV[best_variant]:.3f} in 5-fold CV (rank 1); runner-up {runner_up} \"\n",
    "             f\"({100*(F1[runner_up]-F1[best_variant]):+.1f} pt on validation) is used as a \"\n",
    "             f\"sensitivity check in notebook 04\"))\n",
    "\n",
    "decision = pd.DataFrame(rows, columns=[\"Question\", \"Decision\", \"Evidence\"]).set_index(\"Question\")\n",
    "decision.to_csv(METRIC_DIR / \"preprocessing_decision.csv\")\n",
    "\n",
    "print(\"best on validation :\", comparison.index[0], f\"({F1.iloc[0]:.4f})\")\n",
    "print(\"best on 5-fold CV  :\", cv_table.index[0], f\"({cv_table.cv_f1_mean.iloc[0]:.4f})\")\n",
    "print(\"kept for notebook 04:\", best_variant, \"| runner-up:\", runner_up)\n",
    "decision"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "00075970",
   "metadata": {},
   "source": [
    "### Written decision (for the report)\n",
    "\n",
    "| Question | Decision | Evidence |\n",
    "|---|---|---|\n",
    "| Remove stopwords (negations protected)? | **Yes** | F1 +0.9 pt on validation, +1.9 pt in CV (same sign); the token count of `not` is provably unchanged, so no negation is lost |\n",
    "| Keep pronouns? | **No — not established** | validation +1.6 / +1.1 pt but CV −1.5 / −1.6 pt: the two estimates disagree, the effect is inside the noise. Simpler vocabulary chosen |\n",
    "| Drop the `user` placeholder? | **No** | +1 pt in both estimates, but it empties 58 training and 6 validation tweets and the gain is below the fold-to-fold spread |\n",
    "| Lemmatise? | **Yes** | F1 +3.3 pt on validation and +4.1 pt in CV, vocabulary 18.5k → 14.7k tokens: the only unambiguous gain |\n",
    "| **Final pipeline** | `clean (Emna) → stopwords (negations protected) → lemmatisation` | variant **C**, F1 0.520 validation / 0.501 CV; runner-up **C2** carried to notebook 04 as a sensitivity check |\n",
    "\n",
    "**Known limits of the retained steps** (to state in the report):\n",
    "* WordNet maps `hating → hat`, and without the mandatory verb pass `was → wa`; slang and typos\n",
    "  (`libtards`, `idiottt`) are left unchanged.\n",
    "* Lemmatisation costs ~2× more time than the other steps (2.2 s vs 1.4 s per experiment here — it\n",
    "  would matter much more on a bigger corpus).\n",
    "* Stopword removal can leave a tweet empty in principle; measured here: **0** for variant C\n",
    "  (58 for variant B3, which we rejected).\n",
    "\n",
    "> Sanity anchor: the reference variant A reproduces the guide's check almost exactly\n",
    "> (accuracy 0.744, precision 0.816, recall 0.338, F1 0.478 vs the announced 0.74 / 0.82 / 0.33 / 0.47).\n",
    "> The whole comparison therefore rests on a setup that is known to be correct.\n",
    "\n",
    "## 8 · Hand-over to notebook 04\n",
    "\n",
    "* Interface with the next notebook:\n",
    "  `results/metrics/03_preprocessing_experiments/preprocessing_comparison.csv` — notebook 04 reads\n",
    "  it, picks the best variant **automatically** (no copy-paste of a name), and only changes the\n",
    "  **training rows**: never the text pipeline, never the validation set, never the test set.\n",
    "* Even with the best preprocessing, **recall is 0.39**: three offensive tweets out of five are still\n",
    "  missed (see the confusion matrices). That is a *class imbalance* problem, and it is exactly what\n",
    "  `04_class_balance_experiments.ipynb` attacks."
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.13.16"
  },
  "title": "03_preprocessing_experiments"
 },
 "nbformat": 4,
 "nbformat_minor": 5
}
